In [1]:
import re
from time import time

import spacy
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

2023-06-20 00:10:17.097082: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-06-20 00:10:18.152849: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-06-20 00:10:18.152875: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2023-06-20 00:10:21.195495: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-

In [28]:
def strip_html(textdata):
    
    soup = BeautifulSoup(textdata, "html.parser")
    return soup.get_text()


def clean_text(textdata):
    
    _only_letters_pattern = re.compile(r"[^A-Za-z0-9']+")
    _no_long_numbers_pattern = re.compile(r'\d{3,}')
    _no_multiple_quotes_pattern = re.compile(r"''+")
    _no_multiple_spaces_pattern = re.compile(r"  +")

    for i in range(len(textdata)):
    
        textdata[i] = strip_html(textdata[i])
        textdata[i] = textdata[i].lower()
        textdata[i] = _only_letters_pattern.sub(' ',textdata[i])
        textdata[i] = _no_long_numbers_pattern.sub('', textdata[i])
        textdata[i] = _no_multiple_quotes_pattern.sub("", textdata[i])
        textdata[i] = _no_multiple_spaces_pattern.sub(" ", textdata[i])
        textdata[i] = textdata[i].strip()

    return textdata

def lemmatize(textdata, cores:int = 1):

    start = time()

    nlp = spacy.load('en_core_web_sm')
    docs = nlp.pipe(textdata, batch_size=100, disable=["parser", "ner"], n_process=cores)
    lemmas = []
    texts = []
    
    end = time()

    print(f"Time to lemmatize: {end-start}")

    start = time()

    for doc in docs:
        
        lemma = [doc[i].text + doc[i+1].text if "'" in doc[i+1].text else doc[i].lemma_.lower() for i in range(len(doc)-1) if "'" not in doc[i].text]
        text = [doc[i].text + doc[i+1].text if "'" in doc[i+1].text else doc[i].text for i in range(len(doc)-1) if "'" not in doc[i].text]

        if doc[-1].text[0] != "'":
            lemma.append(doc[-1].lemma_)

        lemmas.append(lemma)
        texts.append(text)
    
    end = time()

    print(f"Time to process: {end-start}")

    return lemmas, texts


def concatenate_lemmas(lemma_output):
    """
    lemma_output: list of lemmas for each sentence
    """
    sentences = []

    for lemmas in lemma_output:
        sentence = ' '.join(lemmas)
        sentences.append(sentence)
    
    return sentences

sentence = ["I haven't seen the movie yet and I know it is 10 times better than this crap."]
cleaned_text = clean_text(sentence)
print(cleaned_text)

["i haven't seen the movie yet and i know it is 10 times better than this crap"]


In [45]:
nlp = spacy.load("en_core_web_sm")
lemmatizer = nlp.get_pipe("lemmatizer")

directory = '/home/kolla/projects/imdb'
df_train = pd.read_csv(f'{directory}/imdb_train.csv')
#df_train = df_train.sample(frac=1).reset_index(drop=True)
df_test = pd.read_csv(f'{directory}/imdb_test.csv')
#df_test = df_test.sample(frac=1).reset_index(drop=True)
train_data = df_train.values.tolist()
test_data = df_test.values.tolist()

X_train = [x[0] for x in train_data]
Y_train = [x[1] for x in train_data]
X_test = [x[0] for x in test_data]
Y_test = [x[1] for x in test_data]

samples = 10000

train_x = X_train[:samples]
test_x = X_test[:samples]
train_y = Y_train[:samples]
test_y = Y_test[:samples]

In [46]:
train_x_clean = clean_text(train_x)
test_x_clean = clean_text(test_x)

train_x_lemmas, train_x_tokens = lemmatize(train_x_clean)
test_x_lemmas, test_x_tokens = lemmatize(test_x_clean)

/home/kolla/anaconda3/envs/forstaenv/lib/python3.8/site-packages/bs4/__init__.py:435: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  warnings.warn(


Time to lemmatize: 0.5631103515625
Time to process: 154.4776382446289
Time to lemmatize: 0.6217305660247803
Time to process: 167.24267888069153


In [47]:
train_x_lemmas_joined = [" ".join(x) for x in train_x_lemmas]
test_x_lemmas_joined = [" ".join(x) for x in test_x_lemmas]

cv = CountVectorizer(binary=True,
                         max_features=5000,
                         min_df=5,
                         max_df=0.8,
                         stop_words='english')

train_x_bin = cv.fit_transform(train_x_lemmas_joined)
test_x_bin = cv.transform(test_x_lemmas_joined)

clf = LogisticRegression().fit(train_x_bin, train_y)
print("Training Accuracy: %s" % clf.score(train_x_bin, train_y))
print("Test Accuracy: %s" % clf.score(test_x_bin, test_y))

Training Accuracy: 0.9898
Test Accuracy: 0.8425


/home/kolla/anaconda3/envs/forstaenv/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [58]:
def allign_tokens_labels_weights(tokens, vocab_weights, sentiment, threshold):
    
    alligned_tokens = []
    alligned_weights = []

    for i in range(1, len(tokens) + 1):

        bigram = tokens[i-1] + " " + tokens[i] if i < len(tokens) else None
        unigram = tokens[i-1]

        if bigram in vocab_weights.keys():
            alligned_tokens.append(bigram)
            alligned_weights.append(vocab_weights[bigram])

        elif unigram in vocab_weights.keys():
            alligned_tokens.append(unigram)
            alligned_weights.append(vocab_weights[unigram])
        
        else:
            alligned_tokens.append(unigram)
            alligned_weights.append(0.0)

    for j in range(1, len(alligned_tokens)-1):
        
        if " " in alligned_tokens[j] and " " not in alligned_tokens[j-1] and alligned_tokens[j-1] in alligned_tokens[j]:
            alligned_tokens[j-1] = "#"
            weight = alligned_weights[j-1]
            alligned_weights[j-1] = "#"
            alligned_weights[j] += weight
            
        elif " " in alligned_tokens[j] and " " in alligned_tokens[j-1] and alligned_tokens[j-1].split(" ")[1] in alligned_tokens[j]:
            alligned_tokens[j-1] = alligned_tokens[j-1].split(" ")[0]
                
        if alligned_tokens[j] in alligned_tokens[j-1]:
            alligned_tokens[j] = "#"
            weight = alligned_weights[j-1]
            alligned_weights[j] = "#"
            alligned_weights[j-1] += weight
    

    alligned_tokens = [token for token in alligned_tokens if token != "#"]
    alligned_weights = [weight for weight in alligned_weights if weight != "#"]

    if sentiment == 1:
        alligned_labels = [2 if x > threshold else 1 if x < -threshold else 0 for x in alligned_weights]
    else:
        alligned_labels = [1 if x > threshold else 2 if x < -threshold else 0 for x in alligned_weights]
    
    
    return alligned_tokens, alligned_weights, alligned_labels

vocabulary = cv.get_feature_names_out()
weights = clf.coef_[0]
vocabulary_weights = {f"{word}" : weight for word, weight in zip(vocabulary, weights)}

# train_x_lemmas
# train_x_tokens
# train_x_lemmas_joined
# train_x_bin
# train_x_clean

x_alligned_tokens, x_alligned_weights, x_alligned_labels = allign_tokens_labels_weights(train_x_lemmas[0], vocabulary_weights, train_y[0], 0.2)


def test_allign_tokens_labels_weights():
    tokens = ["i", "have", "not", "seen", "the", "movie", "yet", "and", "i", "know", "it", "is", "10", "times", "better", "than", "this", "crap"]
    vocab_weights = {"i have": 0.1, "have not": 0.2, "not seen": 0.3, "seen the": 0.4, "the movie": 0.5, "movie yet": 0.6, "yet and": 0.7, "and i": 0.8, "know it": 0.9, "it is": 1.0, "is 10": 1.1, "10 times": 1.2, "times better": 1.3, "better than": 1.4, "than this": 1.5, "this crap": 1.6}
    sentiment = 1
    threshold = 0.2
    tokens, weights, labels = allign_tokens_labels_weights(tokens, vocab_weights, sentiment, threshold)
    print(tokens)
    print(weights)
    print(labels)

    assert tokens == ['i', 'have', 'not', 'seen', 'the', 'movie', 'yet', 'and', 'i', 'know', 'it', 'is', '10', 'times', 'better', 'than', 'this', 'crap']

test_allign_tokens_labels_weights()

['i', 'have', 'not', 'seen', 'the', 'movie', 'yet', 'and i', 'know', 'it', 'is', '10', 'times', 'better', 'than', 'this crap', 'crap']
[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.6, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 0.0]
[0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0]


AssertionError: 

In [55]:
print(x_alligned_tokens)
print(x_alligned_weights)
print(x_alligned_labels)
print(train_y[0])

['i', 'like', 'how', 'this', 'start', 'out', 'feature', 'some', 'decent', 'special', 'effect', 'especially', 'for', 'a', 'film', '50', 'year', 'old', 'there', 'be', 'some', 'pretty', 'impressive', 'scenery', 'however', 'the', 'film', 'bog', 'down', 'fairly', 'early', 'on', 'with', 'some', 'very', 'dumb', 'dialog', 'as', 'the', 'male', 'all', 'try', 'to', 'flirt', 'with', 'anne', 'francis', 'altaira', 'morbius', 'view', 'this', 'in', "the'", '90', 'after', 'long', 'absence', 'it', 'be', 'fun', 'to', 'see', 'francis', 'again', 'an', 'actress', 'who', 'have', 'do', 'mostly', 'television', 'show', 'since', 'this', 'film', 'be', 'release', 'and', 'be', 'still', 'act', 'it', 'also', 'be', 'interesting', 'to', 'see', 'a', 'young', 'look', 'leslie', 'nielsen', 'dr', 'john', 'adam', 'who', 'i', "wouldn't", 'have', 'recognize', 'have', 'it', 'not', 'be', 'for', 'this', 'voice', 'watch', 'half', 'of', 'this', 'movie', 'before', 'the', 'boredom', 'come', 'almost', 'overwhelming', 'and', 'i', 'have

In [43]:
sentence = "I wouldnt say that it is a bad movie but i dont recommend it either"
sentence = "I'm"

nlp = spacy.load('en_core_web_sm')
lemmatizer = nlp.get_pipe("lemmatizer")

doc = nlp(sentence)

print([token.text for token in doc])
print([token.lemma_ for token in doc])
print([token.pos_ for token in doc])
print([token.tag_ for token in doc])

['I', "'m"]
['I', 'be']
['PRON', 'AUX']
['PRP', 'VBP']
